# Duckietown Explore Retrain (Stable Path)

Uses the existing stable trainer: `/content/leworldduckie/src/train.py`.


In [ ]:
# Config
REPO_URL = 'https://github.com/giuliovv/leworldduckie.git'
DATA_URL = 'https://leworldduckie-public-data.s3.us-east-1.amazonaws.com/duckie_explore.h5'
DATA_LOCAL = '/content/duckie_explore.h5'
RUN_ID = 'duckie_explore_retrain'
EPOCHS = 50
BATCH_SIZE = 128


In [ ]:
# Setup
import os, subprocess
subprocess.check_call(['bash','-lc','python3 -m pip install -U pip'])
subprocess.check_call(['bash','-lc','pip install torch torchvision einops boto3 h5py matplotlib'])
subprocess.check_call(['bash','-lc', f'git clone --depth 1 {REPO_URL} /content/leworldduckie || true'])
if not os.path.exists(DATA_LOCAL):
    subprocess.check_call(['bash','-lc', f'wget -O {DATA_LOCAL} "{DATA_URL}"'])
print('data:', DATA_LOCAL, os.path.getsize(DATA_LOCAL))


In [ ]:
# Train (stable src/train.py path)
import os, subprocess
env = os.environ.copy()
env['DATA_PATH'] = DATA_LOCAL
env['S3_DATA_KEY'] = 'data/duckie_explore.h5'
env['N_EPOCHS'] = str(EPOCHS)
env['BATCH_SIZE'] = str(BATCH_SIZE)
cmd = f'cd /content/leworldduckie && python3 src/train.py --run-id {RUN_ID} --epochs {EPOCHS} 2>&1 | tee /content/train_duckie.log'
print(cmd)
ret = subprocess.run(['bash','-lc',cmd], env=env)
if ret.returncode != 0:
    raise RuntimeError(f'training failed: {ret.returncode}; see /content/train_duckie.log')


In [ ]:
# Output location
# Artifacts are uploaded by src/train.py under:
# s3://leworldduckie/training/runs/<RUN_ID>/
print('Done. Check /content/train_duckie.log and S3 run prefix for checkpoints/metrics.')
